In [2]:
import requests
import time
import pandas as pd
import os

In [3]:
def filter_european(score):
    is_european = False
    temp = score.get("samples_training", [])
    if len(temp) > 0:
        ancestry = temp[0].get("ancestry_broad", [])
        if ancestry == "European":
            is_european = True
        else:
            pass
    if not is_european:
        temp = score.get("samples_variants", [])
        if len(temp) > 0:
            ancestry = temp[0].get("ancestry_broad", [])
            if ancestry == "European":
                is_european = True
            else:
                pass
    return is_european

def get_url(score):
    temp = score.get("ftp_harmonized_scoring_files", [])
    if len(temp) > 0:
        url = temp.get("GRCh38", []).get("positions", [])
        return url
    return False

def get_associdated_pgs_ids(score):
    temp = score.get("associated_pgs_ids", [])
    if len(temp) > 0:
        return temp
    return False

In [ ]:
# Read contained icd and description
icd2dsp = pd.read_csv(os.path.join(os.getcwd(), "trait_list_260225.csv"), index_col=None)
icd2dsp = icd2dsp.dropna(subset=['loinc', 'description'])

# generate icd root col
base_url = "https://www.pgscatalog.org/rest/trait/search"

icd2dsp["pgs_ids"] = None
icd2dsp["pgs_api_num"] = None
icd2dsp["pgs_urls"] = None
for idx, row in icd2dsp.iterrows():
    time.sleep(5)
    loinc = row["loinc"] 
    description = row["ontology"]

    # get all pgs ids for this trait
    params = {"term": description}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Failed for {loinc}: {description}")
        continue
    
    data = response.json()
    url = response.url  

    pgs_ids = []
    while True:
        for s in data["results"]:
            temp = get_associdated_pgs_ids(s)
            if temp:
                pgs_ids.extend(temp)

        next_url = data.get("next")
        if not next_url:
            break
        
        
        r = requests.get(next_url)
        if r.status_code != 200:
            print(f"Pagination failed for {loinc}")
            break
        data = r.json()

    pgs_ids = list(set(pgs_ids))
    icd2dsp.at[idx, "pgs_ids"] = pgs_ids
    icd2dsp.at[idx, "pgs_api_num"] = int(len(pgs_ids))
    
    pgs_urls = []
    for pgs_id in pgs_ids:
        temp = "https://www.pgscatalog.org/rest/score/" + pgs_id
        r = requests.get(temp)
        data = r.json()

        # filter european pgs and get corresponding url
        # if filter_european(data):
        #     euro_url = get_url(data)
        #     pgs_urls.extend([euro_url])
        #     time.sleep(0.2)
        
        # get ll pgs`s corresponding url
        temp_url = get_url(data)
        pgs_urls.extend([temp_url])
        time.sleep(0.2)
    
    icd2dsp.at[idx, "pgs_urls"] = pgs_urls
    print(f"Finish scrolling {loinc} with num of pgs:", len(pgs_ids), ", num of pgs urls:", len(pgs_urls))
    time.sleep(0.2)

icd2dsp.to_csv(os.path.join(os.getcwd(),"pgs_id_list_260225.csv"), index=False)

Finish scrolling 1869-7 with num of pgs: 3 , num of pgs urls: 3
Finish scrolling 1869-7 with num of pgs: 1 , num of pgs urls: 1
Finish scrolling 1884-6 with num of pgs: 4 , num of pgs urls: 4
Finish scrolling 704-7 with num of pgs: 14 , num of pgs urls: 14
Finish scrolling 706-2 with num of pgs: 5 , num of pgs urls: 5
Finish scrolling 39156-5 with num of pgs: 125 , num of pgs urls: 125


In [ ]:
import requests
import time

url = "https://www.pgscatalog.org/rest/score/all"

euro_urls = []

while url:
    r = requests.get(url)
    data = r.json()
    
    euro_data = [s for s in data["results"] if filter_european(s)]
    euro_url = [get_url(s) for s in euro_data if get_url(s)]
    euro_urls.extend(euro_url)
    
    url = data["next"]
    time.sleep(0.2)   

print("Total European GRCh38 URLs:", len(euro_urls))

Total European GRCh38 URLs: 2334


In [7]:
import pandas as pd
import os
import ast
icd2dsp = pd.read_csv(os.path.join(os.getcwd(), "pgs_id_list_260225.csv"), index_col=None)
pgs_series = icd2dsp["pgs_ids"].dropna().apply(ast.literal_eval)
all_pgs = pgs_series.explode()
n_unique = all_pgs.nunique()
print(n_unique)

1155


In [6]:
print(icd2dsp)

                             ontology  pgs_num    loinc  \
0      apolipoprotein a 1 measurement        3   1869-7   
1     apolipoprotein a-iv measurement        1   1869-7   
2        apolipoprotein b measurement        4   1884-6   
3                      basophil count        9    704-7   
4   basophil percentage of leukocytes        5    706-2   
..                                ...      ...      ...   
60           triglyceride measurement       75   2571-8   
61              uric acid measurement        2   3084-1   
62       ventricular rate measurement        2   8867-4   
63                    vitamin d level       44  62292-8   
64                    waist-hip ratio       10      NaN   

                                          description  
0   Apolipoprotein A-I [Mass/volume] in Serum or P...  
1   Apolipoprotein A-I [Mass/volume] in Serum or P...  
2   Apolipoprotein B [Mass/volume] in Serum or Plasma  
3    Basophils [#/volume] in Blood by Automated count  
4    Basoph

In [3]:
print(data)

{'id': 'PGS003852', 'name': 'CRC_PRS_EUR_EAS', 'ftp_scoring_file': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/PGS003852.txt.gz', 'ftp_harmonized_scoring_files': {'GRCh37': {'positions': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/Harmonized/PGS003852_hmPOS_GRCh37.txt.gz'}, 'GRCh38': {'positions': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/Harmonized/PGS003852_hmPOS_GRCh38.txt.gz'}}, 'publication': {'id': 'PGP000492', 'title': 'Combining Asian and European genome-wide association studies of colorectal cancer improves risk prediction across racial and ethnic populations.', 'doi': '10.1038/s41467-023-41819-0', 'PMID': 37783704, 'journal': 'Nat Commun', 'firstauthor': 'Thomas M', 'date_publication': '2023-10-02'}, 'matches_publication': True, 'samples_variants': [{'sample_number': 69175, 'sample_cases': None, 'sample_controls': None, 'sample_percent_male': None, 'sample_age': None, 'phenotypin